In [1]:
import requests
import pandas as pd

# global variables
EVENTS_API = "https://api.elections.kalshi.com/trade-api/v2/events" 
HISTORICAL_API = "https://api.elections.kalshi.com/trade-api/v2/historical/markets"


<h3>GET KALSHI DATA

In [82]:

def get_eventtickers_from_series(series_ticker):
    series_params= {
            "series_ticker": series_ticker,
            "status": "settled",
            "limit": 200 #max
        }
    response = requests.get(EVENTS_API, params=series_params)
    events_data = response.json()['events']
    events_df = pd.DataFrame(events_data)
    return events_df['event_ticker']


def get_markets_from_event(event_ticker):
    event_params= {
        "event_ticker": event_ticker,
        "status": "settled",
        "limit": 1000 #max
    }
    response = requests.get(HISTORICAL_API, params=event_params)
    market_data = response.json()['markets']
    market_data_df = pd.DataFrame(market_data)
    #display(market_data_df.head(3)) #we want: ticker, floor_strike, last_price_dollars, volume_fp, yes_ask_dollars

    return market_data_df


event_tickers = get_eventtickers_from_series("KXCPIYOY") #must be all capitals
markets = []
for event_ticker in event_tickers:
    market_data = get_markets_from_event(event_ticker)
    
    if market_data is None or market_data.empty:
        continue
    if 'floor_strike' not in market_data.columns:
        print(f"{event_ticker} — no floor_strike, columns are: {market_data.columns.tolist()}")
        continue
    
    markets.append({
        "event_ticker": event_ticker,
        "settlement_date": market_data['settlement_ts'],
        "markets": market_data[['floor_strike', 'last_price_dollars']]
    })


markets = pd.DataFrame(markets)


CPIYOY-23NOV — no floor_strike, columns are: ['can_close_early', 'close_time', 'created_time', 'event_ticker', 'expected_expiration_time', 'expiration_time', 'expiration_value', 'fractional_trading_enabled', 'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars', 'market_type', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title', 'notional_value_dollars', 'open_interest_fp', 'open_time', 'previous_price_dollars', 'previous_yes_ask_dollars', 'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges', 'response_price_units', 'result', 'rules_primary', 'rules_secondary', 'settlement_timer_seconds', 'settlement_ts', 'settlement_value_dollars', 'status', 'subtitle', 'tick_size', 'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars', 'yes_bid_size_fp', 'yes_sub_title']
CPIYOY-23SEP — no floor_strike, columns are: ['can_close_early', 'close_time', 'created_time', 'event_ticker', 'expected_expiration_time',

In [83]:
display(markets.head())



,event_ticker,settlement_date,markets
0,KXCPIYOY-25DEC,0 2026-01-13T16:36:48.384816Z 1 2026-01-...,floor_strike last_price_dollars 0 ...
1,KXCPIYOY-25NOV,0 2025-12-18T15:34:01.234735Z 1 2025-12-...,floor_strike last_price_dollars 0 ...
2,KXCPIYOY-25OCT,0 2025-11-22T02:04:58.890565Z 1 2025-11-...,floor_strike last_price_dollars 0 ...
3,KXCPIYOY-25SEP,0 2025-10-24T13:12:57.241834Z 1 2025-10-...,floor_strike last_price_dollars 0 ...
4,KXCPIYOY-25AUG,0 2025-09-11T13:43:49.021077Z 1 2025-09-...,floor_strike last_price_dollars 0 ...


<h3>KALSHI -> PREDICTION (FED-MAGIC)

In [84]:



# ── Core calculation ──────────────────────────────────────────────────────────

def build_distribution_one_event(df):
    df = df.copy()

    # Convert to float — API returns strings
    df['floor_strike'] = df['floor_strike'].astype(float)
    df['last_price_dollars'] = df['last_price_dollars'].astype(float)

    df = df.sort_values('floor_strike').reset_index(drop=True)

    # fix 2: compute step once so no NaN on first row
    step = df['floor_strike'].diff().median()

    # fix 3: add bottom bin (probability below lowest strike)
    bottom_bin = pd.DataFrame([{
        'floor_strike': df['floor_strike'].iloc[0] - step,
        'last_price_dollars': 1.0,
        'prob': 1.0 - df['last_price_dollars'].iloc[0]
    }])
    df = pd.concat([bottom_bin, df]).reset_index(drop=True)

    # bin probabilities for middle and top bins
    df['prob'] = df['last_price_dollars'].diff(-1)
    df.loc[df.index[-1], 'prob'] = df['last_price_dollars'].iloc[-1]

    # fix negatives and renormalise
    df['prob'] = df['prob'].clip(lower=0)
    df['prob'] /= df['prob'].sum()

    # fix 2: midpoint using pre-computed step, no NaN
    df['midpoint'] = df['floor_strike'] + step / 2

    return df

def compute_mean(df):
    return (df['midpoint'] * df['prob']).sum()

# Loop over each event, not the whole markets DataFrame at once
results = []
for _, row in markets.iterrows():
    dist = build_distribution_one_event(row['markets'])
    mean = compute_mean(dist)
    results.append({
        "event_ticker": row['event_ticker'],
        "predicted_date": row['settlement_date'].iloc[0],
        "predicted_CPI_MoM": mean,
        "check_sum": dist['prob'].sum()
    })
    # print(f"{row['event_ticker']} — Mean: {mean:.4f}")
   

In [85]:
results_df = pd.DataFrame(results)
display(results_df.tail())
# results_df.to_excel("kalshi_cpi_results.xlsx", index=False)

,event_ticker,predicted_date,predicted_CPI_MoM,check_sum
30,CPIYOY-23APR,2023-05-10T17:05:01.540598Z,2.455155,1.0
31,CPIYOY-23MAR,2023-04-12T17:03:01.034778Z,4.327073,1.0
32,CPIYOY-23FEB,2023-03-14T17:06:00.64294Z,5.251531,1.0
33,CPIYOY-23JAN,2023-02-14T18:10:01.416443Z,5.819597,1.0
34,CPIYOY-22DEC,2023-01-12T19:06:00.956757Z,6.360891,1.0


<h3> AWEL WA NU

In [86]:
import pandas as pd
from fredapi import Fred

FRED_API_KEY = '94318e235b2c36c1a3159affe94b3ffc' # Get one at: https://fredaccount.stlouisfed.org/apikeys
SERIES_ID = 'CPIAUCSL'
fred = Fred(api_key=FRED_API_KEY)

def get_monthly_economic_data(start_date='2021-12-01'):  # start 1 year earlier to allow YoY calc
    series_data = fred.get_series(SERIES_ID, observation_start=start_date)
    df = pd.DataFrame(series_data, columns=['cpi_level'])
    df.index = pd.to_datetime(df.index)
    df = df.resample('MS').mean()
    
    # YoY % change: (current - 12 months ago) / 12 months ago * 100
    df['cpi_yoy'] = df['cpi_level'].pct_change(12) * 100
    
    return df[['cpi_yoy']].dropna()
  


# ── Run ───────────────────────────────────────────────────────────────────────
df = get_monthly_economic_data('2021-12-01')
display(df)


C:\Users\louis\AppData\Local\Temp\ipykernel_23768\1187684722.py:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['cpi_yoy'] = df['cpi_level'].pct_change(12) * 100


,cpi_yoy
2022-12-01,6.404600
2023-01-01,6.327179
2023-02-01,5.957821
2023-03-01,4.917719
2023-04-01,4.950080
2023-05-01,4.131851
2023-06-01,3.070617
2023-07-01,3.287749
2023-08-01,3.722505
2023-09-01,3.687207


In [87]:
def get_release_month(event_date_str: str) -> str:
    """Extract YYYY-MM from a settlement timestamp string."""
    return str(pd.to_datetime(event_date_str).to_period("M"))


def merge_with_actual(results_df: pd.DataFrame, fred_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge Kalshi predictions with actual FRED CPI YoY data.
    Computes error, absolute error and R^2.
    
    results_df : output of the Kalshi loop — needs event_ticker, event_date, mean
    fred_df    : output of get_monthly_economic_data() — index is datetime, column is cpi_yoy
    """
    df = results_df.copy()
    
    # extract release month from settlement timestamp
    df["release_month"] = df["predicted_date"].apply(get_release_month)
    
    # prepare FRED df for merging
    fred = fred_df.copy()
    fred["release_month"] = fred.index.to_period("M").astype(str)
    
    # merge on release_month
    merged = df.merge(fred, on="release_month", how="left")
    
    # errors
    merged["error"]     = (merged["predicted_CPI_MoM"] - merged["cpi_yoy"]).round(4)
    merged["abs_error"] = merged["error"].abs().round(4)
    
    # R^2 — only on rows where we have both prediction and actual
    clean = merged.dropna(subset=["predicted_CPI_MoM", "cpi_yoy"])
    ss_res = ((clean["cpi_yoy"] - clean["predicted_CPI_MoM"]) ** 2).sum()
    ss_tot = ((clean["cpi_yoy"] - clean["cpi_yoy"].mean()) ** 2).sum()
    r2 = 1 - (ss_res / ss_tot)
    
    print(f"R²: {r2:.4f}")
    print(f"MAE: {merged['abs_error'].mean():.4f}%")
    print(f"N:  {len(clean)} events")
    
    return merged[["event_ticker", "predicted_date", "predicted_CPI_MoM", "cpi_yoy", "error", "abs_error"]]


# ── Run ───────────────────────────────────────────────────────────────────────
fred_df = get_monthly_economic_data(start_date='2021-12-01')
evaluation = merge_with_actual(results_df, fred_df)
display(evaluation)

R²: 0.7533
MAE: 0.3332%
N:  35 events


C:\Users\louis\AppData\Local\Temp\ipykernel_23768\1187684722.py:15: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['cpi_yoy'] = df['cpi_level'].pct_change(12) * 100
C:\Users\louis\AppData\Local\Temp\ipykernel_23768\2175990887.py:3: UserWarning: Converting to Period representation will drop timezone information.
  return str(pd.to_datetime(event_date_str).to_period("M"))


,event_ticker,predicted_date,predicted_CPI_MoM,cpi_yoy,error,abs_error
0,KXCPIYOY-25DEC,2026-01-13T16:36:48.384816Z,2.698039,2.391201,0.3068,0.3068
1,KXCPIYOY-25NOV,2025-12-18T15:34:01.234735Z,2.928846,2.653304,0.2755,0.2755
2,KXCPIYOY-25OCT,2025-11-22T02:04:58.890565Z,3.043137,2.696444,0.3467,0.3467
3,KXCPIYOY-25SEP,2025-10-24T13:12:57.241834Z,2.995098,2.729136,0.2660,0.2660
4,KXCPIYOY-25AUG,2025-09-11T13:43:49.021077Z,2.819000,3.022572,-0.2036,0.2036
5,KXCPIYOY-25JUL,2025-08-12T13:09:05.969386Z,2.746000,2.938592,-0.1926,0.1926
6,KXCPIYOY-25JUN,2025-07-15T13:05:41.994613Z,2.585644,2.742618,-0.1570,0.1570
7,KXCPIYOY-25MAY,2025-06-11T14:27:48.483421Z,2.384951,2.680454,-0.2955,0.2955
8,KXCPIYOY-25APR,2025-05-13T12:57:16.086936Z,2.341000,2.377265,-0.0363,0.0363
9,KXCPIYOY-25MAR,2025-04-10T15:02:37.840168Z,2.484259,2.325388,0.1589,0.1589


In [89]:
evaluation.to_excel("kalshi_cpi_results.xlsx", index=False)